### Task 1

In [1]:
import requests
import re
from bs4 import BeautifulSoup
import numpy as np
import pandas as pd


url = f'https://www.amazon.in/HP-S1000-Wireless-Connection-Symmetrical/dp/B0847MDVNB/'
resp = requests.get(url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, 'html.parser')

In [2]:
tags = soup.find_all('div', class_='a-section a-spacing-none a-padding-none')
all_cards = []
for review in tags:
    cards = review.find_all('div', class_='a-section aok-relative', attrs={'data-hook': 'review'})
    all_cards.extend(cards)

In [3]:
reviews=[]
for card in cards:
  review_tag=card.find('div',class_='_Y3Itd_contain-rich-content_2IORW', attrs={'data-hook': "reviewRichContentContainer"})
  review = review_tag.get_text(strip=True) if review_tag else np.nan
  if len(review)>10 and (re.sub(r"[^a-zA-Z0-9\s]", "", review).strip()):
    reviews.append(review)

In [4]:
reviews[0]

"I have an AIO PC and the mouse got damaged after 5 years basically due to my negligence. I didn't check the battery and it leaked and damaged the mouse.Instead of wasting time on repair, I ordered this one. But, in the hurry, I forgot to connect the USB connector provided with the mouse. Obviously, it didn't work. When I went through the operating instructions, I realised my mistake and connected the USB connector to the port. It started working without any issues."

In [5]:
!pip install groq

In [6]:
instructions = '''
You are an expert sentiment analysis assistant.
Your task is to analyze customer reviews, break down your reasoning step-by-step,
and assign a final sentiment label of either 'positive', 'neutral', or 'negative'.
'''

context = '''
 ### EXAMPLES

Review: "The battery lasts for months and it glides smoothly on my desk. Perfect ergonomics."
Reasoning Process:
- Step 1 (Positive): "battery lasts for months", "glides smoothly", "Perfect ergonomics".
- Step 2 (Negative): None.
- Step 3 (Mixed): No contradictions.
- Step 4 (Label): Uniformly high praise for performance and comfort.
Final Label: positive

---

Review: "Tracking is highly accurate for gaming, but the clicks are way too loud for an office environment."
Reasoning Process:
- Step 1 (Positive): "highly accurate for gaming".
- Step 2 (Negative): "clicks are way too loud".
- Step 3 (Mixed): Yes, great tracking but disruptive noise level.
- Step 4 (Label): The positive utility balances out the negative noise issue.
Final Label: neutral

---

Review: "The scroll wheel completely broke after two weeks. Cheap plastic garbage."
Reasoning Process:
- Step 1 (Positive): None.
- Step 2 (Negative): "completely broke after two weeks", "Cheap plastic garbage".
- Step 3 (Mixed): No contradictions. Fully hostile.
- Step 4 (Label): Severe hardware failure and poor quality build.
Final Label: negative

---

### Your Task

Review: {review}
Reasoning Process:
'''

In [7]:
from groq import Groq
from google.colab import userdata
client = Groq(api_key=userdata.get('groq'))
client

In [8]:
results=[]
for i,review in enumerate(reviews):
  chat_completion = client.chat.completions.create(
      messages=[
          {

              "role": "system",
              "content": instructions,
          },
           {
               "role": "user",
               "content": context.format(review=review),
            }],
      model='openai/gpt-oss-120b')

  results.append(chat_completion.choices[0].message.content)

In [12]:
from IPython.display import Markdown
for i,j in enumerate(results):
  print(f'\nResponse for Review {i+1}')
  display(Markdown(j))


Response for Review 1


**Reasoning Process:**

- **Step 1 (Positive cues):**  
  - “It started working without any issues.”  
  - The mouse functions correctly once the USB connector is attached.  
  - No complaints about build quality, performance, or design.

- **Step 2 (Negative cues):**  
  - The reviewer mentions forgetting to connect the USB connector, which caused the mouse not to work initially.  
  - This is a user‑error rather than a flaw of the product itself, and it is quickly resolved.

- **Step 3 (Mixed assessment):**  
  - The only negative element is the temporary inconvenience of the missed connection.  
  - Since the problem is self‑inflicted and the product works perfectly after correction, the overall impression remains favorable.

- **Step 4 (Label decision):**  
  - The review ends on a positive note, emphasizing that the mouse operates without any issues.  
  - The brief negative incident does not outweigh the positive experience.

**Final Label:** **positive**


Response for Review 2


**Reasoning Process**

- **Step 1 (Positive aspects)**  
  - The reviewer notes that *Amazon’s customer service was much better* than the product experience.  
  - The clicks themselves are described as “silent,” which is a positive design intention (even though it is compromised by the shell).

- **Step 2 (Negative aspects)**  
  - **Noise issues:** The surface shell makes a noticeable sound on each click, and the scroll wheel is “very very loud” and of cheap, poor quality.  
  - **Build quality:** The shell feels cheap; the internals are the same low‑cost Chinese boards used by generic brands.  
  - **Reliability problems:** The hardware‑level auto‑sleep feature is “VERY INCONSISTENT,” frequently turning the mouse completely off and failing to reconnect.  
  - **Defectiveness:** The reviewer received a defective unit, and similar failures have been reported by others.  
  - **Service inconvenience:** Repairs require the user to personally take the mouse to an HP service centre, which are “few and far.”  
  - **Overall disappointment:** The reviewer explicitly states the mouse was “terrible” and that HP’s mouse quality has “degraded tremendously,” especially compared with an older HP model that was superior.

- **Step 3 (Mixed assessment)**  
  - The only mitigating positive comment is the good Amazon customer‑service experience, which does not address the product’s core performance or reliability issues.  
  - The overwhelming number of complaints about quality, functionality, and design outweighs the single positive service remark.

- **Step 4 (Label)**  
  - The review is dominated by severe negative feedback about the mouse’s build, reliability, and usability, with only a minor positive note about external customer service.  
  - Therefore, the overall sentiment is **negative**.

**Final Label:** negative


Response for Review 3


...

**Reasoning Process:**

- **Step 1 (Positive):**  
  - “Great value for money.” – Indicates strong cost‑performance satisfaction.  
  - “Fairly light mouse.” – Highlights a desirable physical characteristic.  
  - “Works brilliantly.” – Explicit praise for performance.  
  - “Works out of the box easily.” – Suggests plug‑and‑play convenience and reliability.  
  - “Hence bought a second one.” – The reviewer’s repeat purchase signals high approval.

- **Step 2 (Negative):**  
  - No negative remarks or complaints are present in the review.

- **Step 3 (Mixed):**  
  - There are no contradictory statements; the sentiment is uniformly positive.

- **Step 4 (Label):**  
  - The review consistently praises the product’s value, design, functionality, and ease of use, culminating in a repeat purchase.  
  - **Final Label:** **positive**


Response for Review 9


**Reasoning Process**

**Step 1 – Identify Positive Statements**  
- “The clicks are truly silent, making it perfect for office and late‑night use.” → praises a key feature.  
- “The wireless connection is smooth, setup was instant.” → highlights reliable performance and easy installation.  
- “The mouse feels comfortable with accurate tracking.” → compliments ergonomics and functionality.  
- “The seller provided a great buying experience — the product was delivered promptly, well packaged, and exactly as described.” → positive remarks about order fulfillment and seller professionalism.  
- “Everything was smooth from order to delivery, which is always appreciated.” → reinforces the hassle‑free purchase.  
- “High‑quality mouse at a very reasonable price.” → positive assessment of value.  
- “The seller is reliable and professional. Highly recommended.” → strong endorsement.

**Step 2 – Identify Negative Statements**  
- No negative comments, complaints, or criticisms are present.

**Step 3 – Check for Mixed Sentiment**  
- Since there are no negative points, there is no conflict or mixed sentiment. The review is uniformly positive.

**Step 4 – Assign Final Sentiment Label**  
- The review consists entirely of praise for the product’s features, performance, value, and the seller’s service.  
**Final Label:** **positive**


Response for Review 10


**Reasoning Process**

- **Step 1 (Positive):**  
  - “Great product.” – explicit praise.  
  - “Clicking is silent for normal use” – highlights a desirable feature.  
  - “For me it works perfectly” – indicates overall satisfaction.  
  - “Rest all good.” – reinforces that most aspects meet the reviewer’s expectations.

- **Step 2 (Negative):**  
  - “but has a bit of sound of course when room has no sound.” – notes a minor drawback (some audible click).  
  - “just occasionally mouse pointer jumps the screen.” – points out an intermittent functional issue.

- **Step 3 (Mixed):**  
  - The review contains strong positive statements about overall quality and usability, but it also mentions two shortcomings (audible clicks in a silent room and occasional pointer jumps).  
  - The negatives are not severe enough to outweigh the praise, yet they prevent the sentiment from being wholly positive.

- **Step 4 (Label):**  
  - Because the review mixes clear commendation with minor complaints, the appropriate classification is **neutral**.

**Final Label:** neutral

### Task 2

In [13]:
import time
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define Hyperparameter Space (12 configurations)
# We test combinations of n_estimators (tree count) and max_depth (model complexity)
grid = [
    {"n_estimators": 10, "max_depth": 3},   {"n_estimators": 50, "max_depth": 3},   {"n_estimators": 200, "max_depth": 3},
    {"n_estimators": 10, "max_depth": 10},  {"n_estimators": 50, "max_depth": 10},  {"n_estimators": 200, "max_depth": 10},
    {"n_estimators": 10, "max_depth": 20},  {"n_estimators": 50, "max_depth": 20},  {"n_estimators": 200, "max_depth": 20},
    {"n_estimators": 10, "max_depth": None},{"n_estimators": 50, "max_depth": None},{"n_estimators": 200, "max_depth": None}
]

results = []

# 3. Execute Search Grid
for idx, config in enumerate(grid):
    clf = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42
    )

    # Run cross-validation to gather training and validation metrics
    cv_results = cross_validate(clf, X_train, y_train, cv=5, return_train_score=True, n_jobs=-1)

    start_time = time.time()
    clf.fit(X_train, y_train)
    fit_time = time.time() - start_time

    mean_train_acc = cv_results['train_score'].mean()
    mean_val_acc = cv_results['test_score'].mean()
    overfitting_gap = mean_train_acc - mean_val_acc

    results.append({
        "Config_ID": f"C{idx+1}",
        "n_estimators": config["n_estimators"],
        "max_depth": "None" if config["max_depth"] is None else config["max_depth"],
        "Train_Accuracy": round(mean_train_acc, 4),
        "Val_Accuracy": round(mean_val_acc, 4),
        "Overfitting_Gap": round(overfitting_gap, 4),
        "Training_Time_Sec": round(fit_time, 4)
    })

# Convert to DataFrame and Markdown for LLM consumption
df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

| Config_ID   |   n_estimators | max_depth   |   Train_Accuracy |   Val_Accuracy |   Overfitting_Gap |   Training_Time_Sec |
|:------------|---------------:|:------------|-----------------:|---------------:|------------------:|--------------------:|
| C1          |             10 | 3           |           0.9731 |         0.9385 |            0.0346 |              0.0251 |
| C2          |             50 | 3           |           0.9808 |         0.9429 |            0.0379 |              0.1815 |
| C3          |            200 | 3           |           0.9797 |         0.9495 |            0.0302 |              0.5702 |
| C4          |             10 | 10          |           0.9951 |         0.9319 |            0.0632 |              0.0317 |
| C5          |             50 | 10          |           0.9995 |         0.9538 |            0.0456 |              0.1468 |
| C6          |            200 | 10          |           1      |         0.9626 |            0.0374 |              0.5511 |


In [14]:
system_message='''You are an expert Machine Learning Advisor operating under a Tree-of-Thought (ToT) deliberation framework.
               Your task is to evaluate hyperparameter configuration spaces by branching into architectural choices,
               analyzing trade-offs, and pruning non-optimal pathways to discover the most robust configuration.'''
context_message='''You are provided with a dataset containing hyperparameter grid search results for a Random Forest Classifier.
                    Use a Tree-of-Thought process to discover the optimal configuration.
                    {results}'''

In [15]:
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": context_message.format(results=df_results),
        }
    ],

    model='openai/gpt-oss-120b'
)

print(chat_completion.choices[0].message.content)

## Tree‑of‑Thought (ToT) Walk‑through  

Below is a **structured, branch‑and‑prune** exploration of the 12 grid‑search results.  
At each level we ask a concrete question, generate candidate “branches”, evaluate them with the available metrics, and prune the least‑promising paths.  

---

### 1️⃣ Root Question – *What metric truly matters?*  

| Metric | Why it matters | Target |
|--------|----------------|--------|
| **Validation Accuracy** | Direct estimate of out‑of‑sample performance. | **Maximise** |
| **Overfitting Gap** (Train – Val) | Indicates how much the model is memorising the training set. | **Minimise** (≈ 0.03‑0.04 is acceptable) |
| **Training Time** | Practical constraint for scalability / rapid iteration. | **Minimise** (seconds) |
| **Model Complexity** (depth) | Simpler trees are easier to interpret and less prone to variance. | **Prefer lower depth** if performance is equal |

The **primary objective** is the highest **validation accuracy** while keeping the over‑f